In [ ]:
# Notebook 1: Data Exploration and Validation Dataset Preparation

## Purpose
This notebook explores the annotated corpus and raw comment data to:
1. Understand the data structure and resolve annotation count discrepancies
2. Identify the validation dataset (comments with expert annotations)
3. Explore the raw corpus structure for sampling strategy design
4. Perform data quality checks

## Overview
- **Input**: SFU_constructiveness_toxicity_corpus.csv, gnm_comments.csv, gnm_articles.csv
- **Output**: Clean validation dataset with expert labels, data quality report


In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

base_path = Path('../archive/SOCC')
annotated_path = base_path / 'annotated/constructiveness/SFU_constructiveness_toxicity_corpus.csv'
raw_comments_path = base_path / 'raw/gnm_comments.csv'
raw_articles_path = base_path / 'raw/gnm_articles.csv'


## Section 1: Load and Inspect Annotated Corpus

First, we'll load the annotated corpus and understand its structure. The file appears to have ~5,265 lines, but some fields contain newlines, so we need to use pandas to properly parse it.


In [24]:
df_annotated = pd.read_csv(annotated_path)
print(f"Loaded {len(df_annotated)} rows, {len(df_annotated.columns)} columns")


Loaded 1043 rows, 16 columns


## Section 2: Analyze Annotation Quality

Now we'll check which comments have expert annotations. These are the ground truth labels we'll use for model validation.


In [25]:
print("Non-null value counts:")
print(f"toxicity_level: {df_annotated['toxicity_level'].notna().sum()}")
print(f"expert_is_constructive: {df_annotated['expert_is_constructive'].notna().sum()}")
print(f"expert_comments: {df_annotated['expert_comments'].notna().sum()}")


Non-null value counts:
toxicity_level: 1043
expert_is_constructive: 214
expert_comments: 60


In [26]:
df_validation = df_annotated.drop_duplicates(subset=['comment_counter'], keep='first').copy()

def extract_first_toxicity_level(value):
    if pd.isna(value) or value == '':
        return np.nan
    value_str = str(value)
    if value_str and value_str[0].isdigit():
        return int(value_str[0])
    import re
    match = re.search(r'\d', value_str)
    if match:
        return int(match.group())
    return np.nan

if 'toxicity_level' in df_validation.columns:
    before_unique = df_validation['toxicity_level'].nunique()
    df_validation['toxicity_level'] = df_validation['toxicity_level'].apply(extract_first_toxicity_level)
    after_unique = df_validation['toxicity_level'].nunique()
    print(f"Cleaned toxicity_level: {before_unique} -> {after_unique} unique values")
    print(df_validation['toxicity_level'].value_counts().sort_index())

print(f"\nValidation dataset: {len(df_validation)} comments")
print(f"expert_toxicity_level: {df_validation['expert_toxicity_level'].notna().sum()}")
print(f"expert_is_constructive: {df_validation['expert_is_constructive'].notna().sum()}")



Cleaned toxicity_level: 14 -> 4 unique values
toxicity_level
1    829
2    172
3     35
4      7
Name: count, dtype: int64

Validation dataset: 1043 comments
expert_toxicity_level: 214
expert_is_constructive: 214


In [27]:
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

validation_output_path = output_dir / 'validation_1043_comments.csv'
df_validation.to_csv(validation_output_path, index=False)
print(f"Saved: {validation_output_path} ({len(df_validation):,} rows, {len(df_validation.columns)} columns)")


Saved: ..\data\processed\validation_1043_comments.csv (1,043 rows, 16 columns)


In [28]:
excel_output_path = output_dir / 'validation_1043_comments.xlsx'
try:
    df_validation.to_excel(excel_output_path, index=False, engine='openpyxl')
    print(f"Excel file created: {excel_output_path}")
except ImportError:
    try:
        df_validation.to_excel(excel_output_path, index=False, engine='xlsxwriter')
        print(f"Excel file created: {excel_output_path}")
    except ImportError:
        print("No Excel engine available. Install openpyxl or xlsxwriter")


Excel file created: ..\data\processed\validation_1043_comments.xlsx


## Section 3: Load Raw Corpus Metadata

Now we'll explore the raw comment corpus to understand its structure for the sampling strategy. The file is large (>200MB), so we'll use chunked reading to avoid memory issues.


In [29]:
file_size_mb = raw_comments_path.stat().st_size / (1024 * 1024)
print(f"Raw comments file size: {file_size_mb:.2f} MB")

chunk_iterator = pd.read_csv(raw_comments_path, chunksize=10000)
first_chunk = next(chunk_iterator)
print(f"Columns: {len(first_chunk.columns)}")
print(list(first_chunk.columns))


Raw comments file size: 350.15 MB
Columns: 28
['article_id', 'comment_counter', 'comment_author', 'timestamp', 'post_time', 'comment_text', 'TotalVotes', 'posVotes', 'negVotes', 'vote', 'reactions', 'replies', 'comment_id', 'parentID', 'threadID', 'streamId', 'edited', 'isModerator', 'highlightGroups', 'moderatorEdit', 'descendantsCount', 'threadTimestamp', 'flagCount', 'sender_isSelf', 'sender_loginProvider', 'data_type', 'is_empty', 'status']


In [30]:
total_rows = 0
unique_articles = set()
unique_comment_ids = set()
year_counts = {}
top_level_count = 0
reply_count = 0

chunk_iterator = pd.read_csv(raw_comments_path, chunksize=10000)

for i, chunk in enumerate(chunk_iterator):
    total_rows += len(chunk)
    unique_articles.update(chunk['article_id'].dropna().unique())
    
    if 'comment_id' in chunk.columns:
        unique_comment_ids.update(chunk['comment_id'].dropna().unique())
    
    if 'parentID' in chunk.columns:
        top_level_count += chunk['parentID'].isna().sum()
        reply_count += (chunk['parentID'].notna()).sum()
    
    if 'timestamp' in chunk.columns:
        chunk_dates = pd.to_datetime(chunk['timestamp'], unit='ms', errors='coerce')
        chunk_years = chunk_dates.dt.year.dropna()
        for year in chunk_years:
            year_counts[int(year)] = year_counts.get(int(year), 0) + 1
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} chunks ({total_rows:,} rows)...")

print(f"\nTotal comments: {total_rows:,}")
print(f"Unique articles: {len(unique_articles):,}")
print(f"Unique comment IDs: {len(unique_comment_ids):,}")
print(f"Top-level comments: {top_level_count:,}")
print(f"Reply comments: {reply_count:,}")


Processed 10 chunks (100,000 rows)...
Processed 20 chunks (200,000 rows)...
Processed 30 chunks (300,000 rows)...
Processed 40 chunks (400,000 rows)...
Processed 50 chunks (500,000 rows)...
Processed 60 chunks (600,000 rows)...

Total comments: 663,173
Unique articles: 7,797
Unique comment IDs: 655,430
Top-level comments: 282,682
Reply comments: 380,491


In [31]:
if year_counts:
    sorted_years = sorted(year_counts.keys())
    print("\nComments by year:")
    for year in sorted_years:
        print(f"  {year}: {year_counts[year]:,} comments")
    print(f"\nYear range: {min(sorted_years)} to {max(sorted_years)}")



Comments by year:
  2013: 142,264 comments
  2014: 147,031 comments
  2015: 183,529 comments
  2016: 170,818 comments

Year range: 2013 to 2016


## Section 4: Data Quality Checks

Now we'll check for missing values and understand the structure of key columns needed for sampling.


In [32]:
df_sample = pd.read_csv(raw_comments_path, nrows=100000)

key_columns = ['article_id', 'comment_id', 'comment_text', 'timestamp', 'parentID', 
               'TotalVotes', 'posVotes', 'negVotes', 'comment_author']

missing_stats = df_sample[key_columns].isnull().sum()
missing_pct = (missing_stats / len(df_sample) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_stats,
    'Missing %': missing_pct
})
print("Missing values in key columns (sample of 100k rows):")
print(missing_df[missing_df['Missing Count'] > 0])


Missing values in key columns (sample of 100k rows):
          Missing Count  Missing %
parentID          43692      43.69
posVotes          69181      69.18
negVotes          75556      75.56


In [33]:
if 'TotalVotes' in df_sample.columns:
    print("\nTotalVotes statistics:")
    print(df_sample['TotalVotes'].describe())



TotalVotes statistics:
count    100000.000000
mean          0.973920
std           5.835174
min         -59.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         226.000000
Name: TotalVotes, dtype: float64


In [34]:
if 'comment_text' in df_sample.columns:
    text_lengths = df_sample['comment_text'].str.len()
    print("\nComment text quality (sample of 100k rows):")
    print(f"Non-null text: {df_sample['comment_text'].notna().sum():,}")
    print(f"Average length: {text_lengths.mean():.1f} characters")
    print(f"Median length: {text_lengths.median():.1f} characters")
    empty_text = (df_sample['comment_text'].isna() | (df_sample['comment_text'].str.strip() == '')).sum()
    print(f"Empty/whitespace-only: {empty_text:,} ({empty_text/len(df_sample)*100:.2f}%)")



Comment text quality (sample of 100k rows):
Non-null text: 100,000
Average length: 366.8 characters
Median length: 236.0 characters
Empty/whitespace-only: 0 (0.00%)


## Summary and Next Steps

### Key Findings

1. **Validation Dataset**: We have successfully identified comments with expert annotations that will serve as ground truth for model validation.

2. **Data Structure**: The raw corpus contains 663,173+ comments with metadata including timestamps, article IDs, vote counts, and parent IDs for threading.

3. **Sampling Strategy Requirements**: 
   - Article diversity: Sample across multiple articles
   - Temporal coverage: Ensure representation across 2012-2016
   - Comment type: Balance top-level comments vs replies
   - Engagement levels: Use TotalVotes to create engagement bins

### Outputs

- `validation_1043_comments.csv`: Clean validation dataset with expert labels (saved to `data/processed/`)

### Next Steps

- **Notebook 2**: Use the validation dataset to select and validate toxicity models
- **Notebook 3**: Design and execute stratified sampling strategy using the metadata we've explored


In [35]:
print("Summary:")
print(f"Annotated corpus: {len(df_annotated)} rows")
print(f"Validation dataset: {len(df_validation)} comments")
print(f"Raw corpus: {total_rows:,} comments, {len(unique_articles):,} unique articles")


Summary:
Annotated corpus: 1043 rows
Validation dataset: 1043 comments
Raw corpus: 663,173 comments, 7,797 unique articles


In [36]:
if 'toxicity_level' in df_validation.columns:
    toxicity_counts = df_validation['toxicity_level'].value_counts().sort_index()
    
    counts = {}
    for level in [1, 2, 3, 4]:
        count = toxicity_counts.get(float(level), toxicity_counts.get(level, 0))
        counts[level] = int(count)
    
    missing_count = df_validation['toxicity_level'].isna().sum()
    total_comments = len(df_validation)
    valid_comments = total_comments - missing_count
    
    print("Toxicity Level Counts:")
    print(f"1: {counts[1]:,} ({counts[1]/valid_comments*100:.2f}%)")
    print(f"2: {counts[2]:,} ({counts[2]/valid_comments*100:.2f}%)")
    print(f"3: {counts[3]:,} ({counts[3]/valid_comments*100:.2f}%)")
    print(f"4: {counts[4]:,} ({counts[4]/valid_comments*100:.2f}%)")
    print(f"Missing: {missing_count:,}")
    print(f"Total: {total_comments:,}")


Toxicity Level Counts:
1: 829 (79.48%)
2: 172 (16.49%)
3: 35 (3.36%)
4: 7 (0.67%)
Missing: 0
Total: 1,043
